# Anima Eyes 1B v1.2: the Apache-2.0 line

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/qwen-symbolic-thinking/blob/main/notebooks/anima_eyes_v12.ipynb)

The v1.2 fine-tune of Anima's eyes: [OpenGVLab/InternVL3_5-1B-HF](https://huggingface.co/OpenGVLab/InternVL3_5-1B-HF) (its vision tower, projector, word table and language layers 0-23) with the language layers 24-27 and final norm of [Qwen/Qwen3-0.6B-Base](https://huggingface.co/Qwen/Qwen3-0.6B-Base), Anima's own text encoder (the "last-4 hybrid", built by the lane's first run from both at pinned commits), and the Base's own word table as its output head on the rows of the public dataset [thought-symbolic-image-recaptioned](https://huggingface.co/datasets/AbstractPhil/thought-symbolic-image-recaptioned) (release 5), held near the base model's behavior while it learns them (the general-row KL at weight 1, on general requests the base model answered itself), in one pass over the rows. The Apache-2.0 line trains on the dataset's Apache-2.0 part; the non-commercial line (-nc) on every row of the release; the held-out pictures are the same for both, so their reads compare.

Anima reads this model's final states (after the last norm) in its text-encoder slot. So Anima's slot (the Base's layers 24-27 and final norm) carries no LoRA module and stays frozen in the full fine-tune, and beside every micro-batch four Anima-style prompts (the training rows' own generation prompts, tags and captions) hold the model's final states to the Base's (the slot term at weight 1: the relative squared error per token). Every log line reads the agreement on 24 held Anima-style prompts (slot_held, the centered cosine; the hybrid starts at 0.805).

This lane trains the LoRA again (the first was V12_A1_gkl1_1ep_ae1b_by) on the same rows, with the same recipe, slot term and general rows, and one change: the output head's turn-end row (<|im_end|>, the token every read stops on) trains beside the LoRA pairs (1,024 numbers, PEFT's trainable token rows; the rest of the head stays frozen). The first LoRA wrote its rows and never ended them: its head is the Base's word table, whose turn-end row was never trained, and a LoRA of the lower layers did not raise that token. Anima reads the final states, not the head, so the row does not touch Anima's slot. The base comparisons and the general answers were made by the first lane and are not made again; its pace and memory were measured there, so it runs without a smoke.

The queue runs, in order: the training V12_A1_gkl1_1ep_ae1bt_by and its nine reads. The apache line about 5.6 h of training at 2.0 s a step (estimated).

The training runs with resume on: its whole state saves to the project repo every 2 hours and once more after 23.5 hours of the runtime. If the runtime ends mid-run, open this notebook on a fresh runtime and run it again: the lane continues the training from its saved state. Its progress file (runs/progress/<run>.json in the project repo) says where it stands: `python -m qwen_symbolic_thinking.tools.progress <project repo>`.

Runtime: a GPU (the RTX PRO 6000, 95 GB) and the `HF_TOKEN` secret with write access to the project repo `AbstractPhil/qwen-symbolic-thinking-image-training-private`, where every file of this lane lives (its training sets and inputs first: see the package's docs/registry_and_lanes.md). Run C1, restart the session (Runtime > Restart session), then run C2.


In [ ]:
# C1: install, then Runtime > Restart session
import importlib.util, subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("git+https://github.com/AbstractEyes/qwen-symbolic-thinking@v0.1.74", "transformers>=5.0", "accelerate>=1.0", "peft>=0.21.2")

try:                # Colab's torchao 0.10 breaks PEFT when it builds a LoRA (PEFT wants 0.16 or newer, or none at all)
    import torchao
    _v = tuple(int(x) for x in torchao.__version__.split(".")[:2])
    if _v < (0, 16):
        subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
        print("torchao", torchao.__version__, "removed: PEFT cannot build a LoRA beside it")
    else:
        print("torchao", torchao.__version__, "stays")
except ImportError:
    print("no torchao")

print("installed; now Runtime > Restart session, then run C2")

In [ ]:
# C2: work through the lane's queue (runs/queue_anima_eyes_v12.json in the project repo; new runs can be added while it runs)
import os
os.environ["QST_REPO"] = "AbstractPhil/qwen-symbolic-thinking-image-training-private"   # before the package is imported: every file of this lane lives in this repo
LANE = "anima_eyes_v12"

from qwen_symbolic_thinking import REPO, __version__
from qwen_symbolic_thinking.queue import run_queue

print("package", __version__, "| project repo", REPO, "| lane", LANE)
ran = run_queue(LANE, wait_hours=12)
print("ran:", ran)
